# Module 18 — Week 7 — Bayesian Black-Box Optimisation Capstone

**W6: 3/8 improved (F1 BREAKTHROUGH 0.0880, F2 beat initial best, F5 new high)**

### Policy States
| State | Functions | Action |
|-------|-----------|--------|
| **Momentum** | F1, F2, F5 | Tighten and exploit |
| **Recovery** | F6, F7, F8 | Anchor back to all-time best point |
| **Stuck** | F3, F4 | Change approach |

**New this week:** alpha=0.1 for F4 (noise-aware GP) + Yeo-Johnson Y transform

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm, yeojohnson
from scipy.stats.qmc import LatinHypercube
import warnings, os
warnings.filterwarnings('ignore')

PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-18/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print('Module 18 — Week 7 — Bayesian Black-Box Optimisation')
print('New: alpha=0.1 for F4, Yeo-Johnson transform, policy states')

Module 18 — Week 7 — Bayesian Black-Box Optimisation
New: alpha=0.1 for F4, Yeo-Johnson transform, policy states


In [2]:
submitted_x_w1 = {
    1:[0.020584,0.969910], 2:[0.814691,0.969505], 3:[0.376075,0.370839,0.474761],
    4:[0.369789,0.452786,0.367951,0.448446], 5:[0.241041,0.805036,0.948951,0.905090],
    6:[0.466959,0.356875,0.489683,0.726384,0.125125],
    7:[0.027698,0.531762,0.337094,0.176133,0.361503,0.730849],
    8:[0.192432,0.183093,0.018724,0.036362,0.690267,0.444236,0.081374,0.428967]}
new_y_w1 = {1:1.966e-321,2:0.1292261555216582,3:-0.010707313301147062,
            4:-0.34595283782499875,5:1450.9433021815964,6:-0.3611823990070205,
            7:1.4058168801082682,8:9.8915570907296}

submitted_x_w2 = {
    1:[0.591837,0.591837], 2:[0.000000,1.000000], 3:[0.421053,1.000000,1.000000],
    4:[0.909548,0.568955,0.762175,0.811807], 5:[0.204881,0.877830,0.879582,0.870578],
    6:[0.851439,0.906254,0.506372,0.594105,0.708147],
    7:[0.097054,0.432660,0.338116,0.122619,0.296117,0.886436],
    8:[0.076274,0.101214,0.383035,0.338493,0.113685,0.882235,0.615428,0.796463]}
new_y_w2 = {1:0.00028209052469858225,2:0.1709619176069506,3:-0.48304244384724265,
            4:-26.59459580774249,5:1192.2995655092311,6:-1.9259411859252866,
            7:1.2030170341293975,8:9.0382459830856}

submitted_x_w3 = {
    1:[0.980000,0.980000], 2:[1.000000,0.306122], 3:[1.000000,0.000000,0.684211],
    4:[0.985601,0.686679,0.243615,0.798556], 5:[0.204881,0.877830,0.879582,0.870578],
    6:[0.061416,0.762464,0.106527,0.271402,0.782742],
    7:[0.067189,0.412831,0.295130,0.070570,0.412599,0.616173],
    8:[0.682757,0.427203,0.591529,0.734064,0.514947,0.813984,0.722156,0.615073]}
new_y_w3 = {1:2.665897212344236e-174,2:-0.042550557700427774,3:-0.1840890683677661,
            4:-26.07041694623693,5:1192.2995655092311,6:-2.508952125110497,
            7:1.2533263563752521,8:7.5792591902086}

submitted_x_w4 = {
    1:[0.278296,0.020000], 2:[0.685269,0.947006], 3:[0.403468,0.441923,0.497061],
    4:[0.352971,0.651614,0.805417,0.616108], 5:[0.167299,0.881015,0.978872,0.954244],
    6:[0.334649,0.293944,0.500782,0.769829,0.074923],
    7:[0.206363,0.281987,0.389442,0.281544,0.218827,0.711599],
    8:[0.095545,0.327238,0.051339,0.269531,0.555763,0.417489,0.285113,0.613881]}
new_y_w4 = {1:-2.6647756688938686e-133,2:0.14705786268424045,3:-0.022992940111015336,
            4:-0.1283640964538999,5:2496.347187728138,6:-0.38592078647528016,
            7:2.6705394912160187,8:9.8967959631939}

submitted_x_w5 = {
    1:[0.580092,0.683225], 2:[0.702813,0.926626], 3:[0.365086,0.316421,0.471038],
    4:[0.344700,0.645505,0.791987,0.622463], 5:[0.139557,0.911522,0.979905,0.977049],
    6:[0.417831,0.356959,0.468069,0.668531,0.039515],
    7:[0.384097,0.122113,0.444891,0.357064,0.147383,0.783086],
    8:[0.089787,0.068251,0.180968,0.327284,0.766207,0.653365,0.174832,0.499246]}
new_y_w5 = {1:0.00008112997850454906,2:0.5833602539566602,3:-0.018707796769607724,
            4:-13.979947691578896,5:2941.854350298978,6:-0.29985775692426564,
            7:1.660798293687705,8:9.9275118625839}

submitted_x_w6 = {
    1:[0.653384,0.652924], 2:[0.704856,0.921380], 3:[0.151659,0.826046,0.622768],
    4:[0.291709,0.714786,0.911879,0.664486], 5:[0.102387,0.951309,0.977662,0.978193],
    6:[0.394360,0.399099,0.411100,0.595076,0.020599],
    7:[0.027785,0.208441,0.270801,0.266138,0.195016,0.698660],
    8:[0.029320,0.285338,0.223021,0.041430,0.625367,0.719031,0.033888,0.797446]}
new_y_w6 = {1:0.0880341468685302,2:0.6478060146282238,3:-0.0867832750698683,
            4:-21.252967893168208,5:3303.918327634855,6:-0.5181323257010382,
            7:2.1498244177996053,8:9.8116383300479}

all_time_best = {
    1:(0.0880341468685302,'W6'), 2:(0.6478060146282238,'W6'),
    3:(-0.010707313301147062,'W1'), 4:(-0.1283640964538999,'W4'),
    5:(3303.918327634855,'W6'), 6:(-0.29985775692426564,'W5'),
    7:(2.6705394912160187,'W4'), 8:(9.9275118625839,'W5')}

print('All historical data loaded — W1 through W6')
print('\nAll-time bests going into W7:')
for i in range(1,9):
    v,w = all_time_best[i]
    print(f'  F{i}: {v:.4e}  ({w})')

All historical data loaded — W1 through W6

All-time bests going into W7:
  F1: 8.8034e-02  (W6)
  F2: 6.4781e-01  (W6)
  F3: -1.0707e-02  (W1)
  F4: -1.2836e-01  (W4)
  F5: 3.3039e+03  (W6)
  F6: -2.9986e-01  (W5)
  F7: 2.6705e+00  (W4)
  F8: 9.9275e+00  (W5)


In [3]:
base_path = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'
descriptions = {1:'Radiation Detection',2:'Noisy ML Model',3:'Drug Discovery',
                4:'Warehouse Placement',5:'Chemical Yield (STAR)',6:'Cake Recipe',
                7:'ML Hyperparameters',8:'Complex 8D'}

data = {}
for i in range(1,9):
    X0 = np.load(f'{base_path}function_{i}/initial_inputs.npy')
    Y0 = np.load(f'{base_path}function_{i}/initial_outputs.npy')
    X_all = np.vstack([X0,
        np.array(submitted_x_w1[i]).reshape(1,-1),
        np.array(submitted_x_w2[i]).reshape(1,-1),
        np.array(submitted_x_w3[i]).reshape(1,-1),
        np.array(submitted_x_w4[i]).reshape(1,-1),
        np.array(submitted_x_w5[i]).reshape(1,-1),
        np.array(submitted_x_w6[i]).reshape(1,-1)])
    Y_all = np.concatenate([Y0,[new_y_w1[i]],[new_y_w2[i]],[new_y_w3[i]],
                            [new_y_w4[i]],[new_y_w5[i]],[new_y_w6[i]]])
    data[i] = {'X':X_all,'Y':Y_all}

print(f'{"Fn":<4} {"Description":<24} {"N":<5} {"Dim":<5} {"All-time Best Y"}')
print('-'*65)
for i in range(1,9):
    Y=data[i]['Y']; dim=data[i]['X'].shape[1]
    print(f'F{i:<3} {descriptions[i]:<24} {len(Y):<5} {dim:<5} {Y.max():.6e}')
print('\n16 observations per function (10 initial + W1-W6)')

Fn   Description              N     Dim   All-time Best Y
-----------------------------------------------------------------
F1   Radiation Detection      16    2     8.803415e-02
F2   Noisy ML Model           16    2     6.478060e-01
F3   Drug Discovery           21    3     -1.070731e-02
F4   Warehouse Placement      36    4     -1.283641e-01
F5   Chemical Yield (STAR)    26    4     3.303918e+03
F6   Cake Recipe              26    5     -2.998578e-01
F7   ML Hyperparameters       36    6     2.670539e+00
F8   Complex 8D               46    8     9.927512e+00

16 observations per function (10 initial + W1-W6)


In [4]:
BOUND_LO, BOUND_HI = 0.02, 0.98

all_submitted_X = {i:[
    np.array(submitted_x_w1[i]), np.array(submitted_x_w2[i]),
    np.array(submitted_x_w3[i]), np.array(submitted_x_w4[i]),
    np.array(submitted_x_w5[i]), np.array(submitted_x_w6[i])]
    for i in range(1,9)}

def check_duplicate(next_x, func_num, threshold=0.015):
    return any(np.linalg.norm(next_x-x) < threshold for x in all_submitted_X[func_num])

def expected_improvement(mu, sigma, best_y_t, xi=0.01):
    imp = mu - best_y_t - xi
    Z   = imp / (sigma + 1e-9)
    ei  = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

def gp_predict_scalar(gp, x):
    return float(gp.predict(x.reshape(1,-1)).ravel()[0])

def build_search_grid(dim, trust_center=None, trust_radius=None, use_lhs=False, n=50000):
    N = n
    if trust_center is not None and trust_radius is not None:
        lo = np.clip(trust_center-trust_radius, BOUND_LO, BOUND_HI)
        hi = np.clip(trust_center+trust_radius, BOUND_LO, BOUND_HI)
        s  = LatinHypercube(d=dim, seed=42).random(n=N)
        return lo+s*(hi-lo), f'Trust-LHS r={trust_radius:.2f} {dim}D {N:,}pts'
    elif dim == 2:
        g = np.linspace(BOUND_LO, BOUND_HI, 224)
        XX, YY = np.meshgrid(g, g)
        return np.column_stack([XX.ravel(), YY.ravel()]), '224x224 grid ~50k pts'
    elif use_lhs:
        s = LatinHypercube(d=dim, seed=42).random(n=N)
        return BOUND_LO+s*(BOUND_HI-BOUND_LO), f'LHS {N:,}pts {dim}D'
    else:
        np.random.seed(42)
        return np.random.uniform(BOUND_LO, BOUND_HI, (N,dim)), f'Random {N:,}pts {dim}D'

def transform_y(Y, method='log'):
    if method == 'log':
        return np.log(np.abs(Y)+1e-300)*np.sign(Y+1e-300)
    elif method == 'yeojohnson':
        Y_t, _ = yeojohnson(Y)
        return Y_t
    return Y

def analyse_function_w7(func_num, beta_ucb=2.0, use_ei=True,
                         use_lhs=False, trust_center=None, trust_radius=None,
                         xi=0.01, length_scale=0.2, alpha=1e-6,
                         y_transform='log', policy='explore'):
    X, Y = data[func_num]['X'], data[func_num]['Y']
    dim      = X.shape[1]
    best_idx = np.argmax(Y)
    best_X   = X[best_idx] if trust_center is None else trust_center
    best_Y   = Y[best_idx]

    print(f'\n{"="*68}')
    print(f'F{func_num} {descriptions[func_num]} | Dim={dim} N={len(Y)} BestY={best_Y:.4e} | Policy: {policy.upper()}')
    print(f'Best X* = [{" ".join(f"{v:.4f}" for v in X[best_idx])}]')
    print(f'W6 result = {new_y_w6[func_num]:.4e}')
    print('='*68)

    X_grid, grid_info = build_search_grid(
        dim, trust_center=best_X if trust_radius else None,
        trust_radius=trust_radius, use_lhs=use_lhs)
    print(f'  [Grid] {grid_info}')

    Y_t = transform_y(Y, method=y_transform)
    print(f'  [Transform] {y_transform} | range [{Y_t.min():.3f}, {Y_t.max():.3f}]')

    kernel = Matern(length_scale=length_scale, nu=2.5)
    gp     = GaussianProcessRegressor(kernel=kernel, alpha=alpha,
                                       n_restarts_optimizer=3, normalize_y=True)
    gp.fit(X, Y_t)
    print(f'  [GP] alpha={alpha} | {gp.kernel_}')

    mu, sigma = gp.predict(X_grid, return_std=True)
    best_y_t  = float(transform_y(np.array([best_Y]), method=y_transform)[0])

    ucb   = mu + beta_ucb*sigma;  x_ucb = X_grid[np.argmax(ucb)]
    ei    = expected_improvement(mu, sigma, best_y_t, xi=xi)
    x_ei  = X_grid[np.argmax(ei)]

    print(f'  [UCB] b={beta_ucb} => [{" ".join(f"{v:.4f}" for v in x_ucb)}]')
    print(f'  [EI]  xi={xi}  => [{" ".join(f"{v:.4f}" for v in x_ei)}]')

    mu_u = gp_predict_scalar(gp, x_ucb)
    mu_e = gp_predict_scalar(gp, x_ei)
    next_x, winner = (x_ei,'EI') if (use_ei and mu_e >= mu_u) else (x_ucb,'UCB')
    print(f'  [Ensemble] UCB={mu_u:.4f} EI={mu_e:.4f} => {winner}')

    if check_duplicate(next_x, func_num):
        np.random.seed(99)
        next_x = np.clip(next_x+np.random.uniform(-0.03,0.03,dim), BOUND_LO, BOUND_HI)
        print(f'  [Dup] Duplicate — perturbed')

    print(f'  [Dist] {np.linalg.norm(next_x-X[best_idx]):.4f} from best')
    portal = '-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> SUBMIT F{func_num}: {portal} <<<')
    return next_x, portal

print('Helpers ready — W7 (alpha tuning, Yeo-Johnson, policy states)')

Helpers ready — W7 (alpha tuning, Yeo-Johnson, policy states)


In [5]:
# F1 — MOMENTUM — W6: 0.0880 (312x improvement, biggest breakthrough)
W6_BEST_X1 = np.array([0.653384, 0.652924])
next_x1, portal1 = analyse_function_w7(
    func_num=1, beta_ucb=0.8, use_ei=True,
    trust_center=W6_BEST_X1, trust_radius=0.12,
    xi=0.001, policy='momentum')


F1 Radiation Detection | Dim=2 N=16 BestY=8.8034e-02 | Policy: MOMENTUM
Best X* = [0.6534 0.6529]
W6 result = 8.8034e-02
  [Grid] Trust-LHS r=0.12 2D 50,000pts
  [Transform] log | range [-690.776, 305.264]
  [GP] alpha=1e-06 | Matern(length_scale=0.346, nu=2.5)
  [UCB] b=0.8 => [0.5336 0.5337]
  [EI]  xi=0.001  => [0.5336 0.5337]
  [Ensemble] UCB=-3.8594 EI=-3.8594 => EI
  [Dist] 0.1690 from best

  >>> SUBMIT F1: 0.533625-0.533688 <<<


In [6]:
# F2 — MOMENTUM — W6: 0.6478 (beat initial best 0.611)
W6_BEST_X2 = np.array([0.704856, 0.921380])
next_x2, portal2 = analyse_function_w7(
    func_num=2, beta_ucb=0.3, use_ei=True,
    trust_center=W6_BEST_X2, trust_radius=0.08,
    xi=0.005, policy='momentum')


F2 Noisy ML Model | Dim=2 N=16 BestY=6.4781e-01 | Policy: MOMENTUM
Best X* = [0.7049 0.9214]
W6 result = 6.4781e-01
  [Grid] Trust-LHS r=0.08 2D 50,000pts
  [Transform] log | range [-3.768, 4.279]
  [GP] alpha=1e-06 | Matern(length_scale=0.0145, nu=2.5)
  [UCB] b=0.3 => [0.6923 0.9201]
  [EI]  xi=0.005  => [0.6925 0.9195]
  [Ensemble] UCB=0.7345 EI=0.7293 => UCB
  [Dup] Duplicate — perturbed
  [Dist] 0.0030 from best

  >>> SUBMIT F2: 0.702665-0.919352 <<<


In [7]:
# F3 — STUCK — Return to W1 neighbourhood (best is -0.011, W6 gave -0.087)
W1_BEST_X3 = np.array([0.376075, 0.370839, 0.474761])
next_x3, portal3 = analyse_function_w7(
    func_num=3, beta_ucb=1.2, use_ei=True, use_lhs=True,
    trust_center=W1_BEST_X3, trust_radius=0.15,
    xi=0.001, policy='stuck')


F3 Drug Discovery | Dim=3 N=21 BestY=-1.0707e-02 | Policy: STUCK
Best X* = [0.3761 0.3708 0.4748]
W6 result = -8.6783e-02
  [Grid] Trust-LHS r=0.15 3D 50,000pts
  [Transform] log | range [0.728, 4.537]
  [GP] alpha=1e-06 | Matern(length_scale=1e-05, nu=2.5)
  [UCB] b=1.2 => [0.2672 0.4727 0.5131]
  [EI]  xi=0.001  => [0.2672 0.4727 0.5131]
  [Ensemble] UCB=2.5556 EI=2.5556 => EI
  [Dist] 0.1540 from best

  >>> SUBMIT F3: 0.267218-0.472728-0.513126 <<<


In [8]:
# F4 — STUCK — New approach: alpha=0.1 + Yeo-Johnson transform
# alpha=0.1 tells GP not to fully trust noisy observations
# Yeo-Johnson handles extreme negative range better than log transform
W4_BEST_X4 = np.array([0.352971, 0.651614, 0.805417, 0.616108])
next_x4, portal4 = analyse_function_w7(
    func_num=4, beta_ucb=1.5, use_ei=True, use_lhs=True,
    trust_center=W4_BEST_X4, trust_radius=0.20,
    xi=0.01, alpha=0.1, y_transform='yeojohnson', policy='stuck')


F4 Warehouse Placement | Dim=4 N=36 BestY=-1.2836e-01 | Policy: STUCK
Best X* = [0.3530 0.6516 0.8054 0.6161]
W6 result = -2.1253e+01
  [Grid] Trust-LHS r=0.20 4D 50,000pts
  [Transform] yeojohnson | range [-27.417, -0.128]
  [GP] alpha=0.1 | Matern(length_scale=0.572, nu=2.5)
  [UCB] b=1.5 => [0.3414 0.4859 0.6063 0.4785]
  [EI]  xi=0.01  => [0.3414 0.4859 0.6063 0.4785]
  [Ensemble] UCB=-2.2280 EI=-2.2280 => EI
  [Dist] 0.2936 from best

  >>> SUBMIT F4: 0.341374-0.485949-0.606273-0.478470 <<<


In [9]:
# F5 — MOMENTUM — W6: 3303.92 (keeps climbing)
W6_BEST_X5 = np.array([0.102387, 0.951309, 0.977662, 0.978193])
next_x5, portal5 = analyse_function_w7(
    func_num=5, beta_ucb=0.05, use_ei=True, use_lhs=True,
    trust_center=W6_BEST_X5, trust_radius=0.04,
    xi=0.01, policy='momentum')


F5 Chemical Yield (STAR) | Dim=4 N=26 BestY=3.3039e+03 | Policy: MOMENTUM
Best X* = [0.1024 0.9513 0.9777 0.9782]
W6 result = 3.3039e+03
  [Grid] Trust-LHS r=0.04 4D 50,000pts
  [Transform] log | range [-2.181, 8.103]
  [GP] alpha=1e-06 | Matern(length_scale=0.46, nu=2.5)
  [UCB] b=0.05 => [0.0720 0.9770 0.9798 0.9796]
  [EI]  xi=0.01  => [0.1418 0.9782 0.9799 0.9740]
  [Ensemble] UCB=8.1488 EI=8.0812 => UCB
  [Dist] 0.0399 from best

  >>> SUBMIT F5: 0.071951-0.977037-0.979767-0.979593 <<<


In [10]:
# F6 — RECOVERY — Lost W5 best in W6. Anchor back to W5 point.
W5_BEST_X6 = np.array([0.417831, 0.356959, 0.468069, 0.668531, 0.039515])
next_x6, portal6 = analyse_function_w7(
    func_num=6, beta_ucb=0.8, use_ei=True, use_lhs=True,
    trust_center=W5_BEST_X6, trust_radius=0.12,
    xi=0.001, policy='recovery')


F6 Cake Recipe | Dim=5 N=26 BestY=-2.9986e-01 | Policy: RECOVERY
Best X* = [0.4178 0.3570 0.4681 0.6685 0.0395]
W6 result = -5.1813e-01


  [Grid] Trust-LHS r=0.12 5D 50,000pts
  [Transform] log | range [-0.944, 1.204]
  [GP] alpha=1e-06 | Matern(length_scale=0.302, nu=2.5)


  [UCB] b=0.8 => [0.4414 0.2799 0.5149 0.6846 0.0208]
  [EI]  xi=0.001  => [0.4414 0.2799 0.5149 0.6846 0.0208]
  [Ensemble] UCB=1.2871 EI=1.2871 => EI
  [Dist] 0.0964 from best

  >>> SUBMIT F6: 0.441360-0.279908-0.514886-0.684556-0.020780 <<<


In [11]:
# F7 — RECOVERY — Best is W4 (2.671). Anchor back to W4 best.
W4_BEST_X7 = np.array([0.206363, 0.281987, 0.389442, 0.281544, 0.218827, 0.711599])
next_x7, portal7 = analyse_function_w7(
    func_num=7, beta_ucb=1.2, use_ei=True, use_lhs=True,
    trust_center=W4_BEST_X7, trust_radius=0.20,
    xi=0.01, policy='recovery')


F7 ML Hyperparameters | Dim=6 N=36 BestY=2.6705e+00 | Policy: RECOVERY
Best X* = [0.2064 0.2820 0.3894 0.2815 0.2188 0.7116]
W6 result = 2.1498e+00


  [Grid] Trust-LHS r=0.20 6D 50,000pts
  [Transform] log | range [-5.914, 0.982]
  [GP] alpha=1e-06 | Matern(length_scale=0.61, nu=2.5)


  [UCB] b=1.2 => [0.0446 0.2635 0.4804 0.3554 0.2937 0.7079]
  [EI]  xi=0.01  => [0.2552 0.2723 0.2537 0.2385 0.2372 0.6584]
  [Ensemble] UCB=0.6392 EI=0.8437 => EI
  [Dist] 0.1611 from best

  >>> SUBMIT F7: 0.255243-0.272333-0.253655-0.238536-0.237243-0.658362 <<<


In [12]:
# F8 — RECOVERY — Best is W5 (9.9275). Anchor back to W5 best.
W5_BEST_X8 = np.array([0.089787, 0.068251, 0.180968, 0.327284, 0.766207, 0.653365, 0.174832, 0.499246])
next_x8, portal8 = analyse_function_w7(
    func_num=8, beta_ucb=1.2, use_ei=True, use_lhs=True,
    trust_center=W5_BEST_X8, trust_radius=0.25,
    xi=0.01, policy='recovery')


F8 Complex 8D | Dim=8 N=46 BestY=9.9275e+00 | Policy: RECOVERY
Best X* = [0.0898 0.0683 0.1810 0.3273 0.7662 0.6534 0.1748 0.4992]
W6 result = 9.8116e+00


  [Grid] Trust-LHS r=0.25 8D 50,000pts
  [Transform] log | range [1.721, 2.295]
  [GP] alpha=1e-06 | Matern(length_scale=1.32, nu=2.5)


  [UCB] b=1.2 => [0.3063 0.3071 0.1098 0.3615 0.6694 0.4174 0.1752 0.2744]
  [EI]  xi=0.01  => [0.3063 0.3071 0.1098 0.3615 0.6694 0.4174 0.1752 0.2744]
  [Ensemble] UCB=2.3007 EI=2.3007 => EI
  [Dist] 0.4752 from best

  >>> SUBMIT F8: 0.306263-0.307104-0.109816-0.361473-0.669399-0.417361-0.175248-0.274400 <<<


In [13]:
print('='*70)
print('WEEK 7 — MODULE 18 — PORTAL SUBMISSION STRINGS')
print('='*70)
portals = {1:portal1,2:portal2,3:portal3,4:portal4,
           5:portal5,6:portal6,7:portal7,8:portal8}
for i in range(1,9):
    print(f'F{i}: {portals[i]}')
print()
policies = {1:'MOMENTUM',2:'MOMENTUM',3:'STUCK',4:'STUCK',
            5:'MOMENTUM',6:'RECOVERY',7:'RECOVERY',8:'RECOVERY'}
print('Policy states:')
for i in range(1,9):
    v,w = all_time_best[i]
    print(f'  F{i} [{policies[i]}]: best={v:.4e} ({w})')

WEEK 7 — MODULE 18 — PORTAL SUBMISSION STRINGS
F1: 0.533625-0.533688
F2: 0.702665-0.919352
F3: 0.267218-0.472728-0.513126
F4: 0.341374-0.485949-0.606273-0.478470
F5: 0.071951-0.977037-0.979767-0.979593
F6: 0.441360-0.279908-0.514886-0.684556-0.020780
F7: 0.255243-0.272333-0.253655-0.238536-0.237243-0.658362
F8: 0.306263-0.307104-0.109816-0.361473-0.669399-0.417361-0.175248-0.274400

Policy states:
  F1 [MOMENTUM]: best=8.8034e-02 (W6)
  F2 [MOMENTUM]: best=6.4781e-01 (W6)
  F3 [STUCK]: best=-1.0707e-02 (W1)
  F4 [STUCK]: best=-1.2836e-01 (W4)
  F5 [MOMENTUM]: best=3.3039e+03 (W6)
  F6 [RECOVERY]: best=-2.9986e-01 (W5)
  F7 [RECOVERY]: best=2.6705e+00 (W4)
  F8 [RECOVERY]: best=9.9275e+00 (W5)


In [14]:
weekly_results = {
    1:[new_y_w1[i] for i in range(1,9)],
    2:[new_y_w2[i] for i in range(1,9)],
    3:[new_y_w3[i] for i in range(1,9)],
    4:[new_y_w4[i] for i in range(1,9)],
    5:[new_y_w5[i] for i in range(1,9)],
    6:[new_y_w6[i] for i in range(1,9)],
}
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Capstone Progress — Weekly Query Value vs Running Best (W1-W6)', fontsize=14, fontweight='bold')
for idx, fn in enumerate(range(1,9)):
    ax = axes[idx//4][idx%4]
    weeks = [1,2,3,4,5,6]
    vals  = [weekly_results[w][fn-1] for w in weeks]
    running_best = [max(vals[:w]) for w in range(1,len(vals)+1)]
    ax.plot(weeks, vals, 'o--', color='steelblue', label='Weekly query', alpha=0.7)
    ax.plot(weeks, running_best, 's-', color='darkorange', linewidth=2, label='Running best')
    ax.set_title(f'F{fn}: {descriptions[fn]}', fontsize=9)
    ax.set_xlabel('Week'); ax.set_ylabel('Output')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3); ax.set_xticks([1,2,3,4,5,6])
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'w6_progress_analysis.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Plot saved to: {plot_path}')

Plot saved to: /Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-18/plots/w6_progress_analysis.png
